In [1]:
"""
Fine-tune Llama for Physiotherapy with Long Narratives & Music Integration
Creates immersive motivational stories with music cues

Requirements:
pip install transformers peft datasets torch
"""

import os
import json
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

MODEL_OPTIONS = {
    'tiny': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'small': 'meta-llama/Llama-3.2-1B-Instruct',
}

SELECTED_MODEL = 'tiny'
model_name = MODEL_OPTIONS[SELECTED_MODEL]
USE_CPU_ONLY = True
MAX_LENGTH = 768  # Increased for longer narratives
NUM_EPOCHS = 3
BATCH_SIZE = 1
LEARNING_RATE = 2e-4
MAX_STEPS = 50

print("="*60)
print("Enhanced Physiotherapy Fine-tuning")
print("Long Narratives + Music Integration")
print("="*60)
print(f"Model: {model_name}")
print(f"Max length: {MAX_LENGTH} tokens")
print("="*60)

# ============================================================================
# STEP 1: CREATE ENHANCED TRAINING DATA
# ============================================================================

def create_training_data():
    """Create training data with long narratives and music cues"""
    
    training_data = [
        {
            "instruction": "Create a comprehensive daily physiotherapy plan with an immersive motivational narrative that includes music cues from the patient's playlist.",
            "patient_history": """John Smith is a 45-year-old male software engineer who has been struggling with chronic lower back pain for the past two years. His condition developed gradually due to prolonged sitting at his desk job, often working 10-12 hour days without proper breaks. The pain started as occasional discomfort but has progressively worsened to a constant dull ache that intensifies after sitting for more than 30 minutes. John describes the pain as a deep, burning sensation that radiates from his lower lumbar region down to his left hip. He rates his current pain level at 6/10, which increases to 8/10 by the end of his workday. John has tried basic stretching exercises in the past but found it difficult to maintain consistency due to his busy schedule and initial lack of visible results. He's motivated to improve because he wants to be able to play actively with his two young children (ages 6 and 8) without discomfort, and he's noticed that his wife has been concerned about his declining mobility. John's sedentary lifestyle has also contributed to weight gain (15 pounds over the past year) and reduced core strength. Previous treatments included occasional visits to a massage therapist and over-the-counter pain medication, but these provided only temporary relief. He's now committed to a structured physiotherapy program and is willing to dedicate 45 minutes daily to his recovery.""",
            "spotify_playlist": "Eye of the Tiger by Survivor, Stronger by Kanye West, Don't Stop Believin' by Journey, Hall of Fame by The Script, Lose Yourself by Eminem",
            "output": """**DAILY EXERCISE SCHEDULE:**

**Morning Session (7:30 AM - 8:00 AM):**
1. Cat-Cow Stretch - 3 minutes, 15 slow repetitions, focus on spinal articulation
2. Pelvic Tilts - 4 sets of 15 repetitions, hold each tilt for 3 seconds
3. Gentle walking around the house - 10 minutes at comfortable pace
4. Dead Bug Exercise - 3 sets of 8 repetitions per side
5. Knee-to-Chest Stretch - Hold 30 seconds each leg, 3 repetitions

**Afternoon Session (1:00 PM - 1:20 PM):**
6. Seated Spinal Twist (at desk) - 45 seconds each side, 3 repetitions
7. Standing Hip Flexor Stretch - 40 seconds each side, 3 repetitions
8. Bird-Dog Exercise - 4 sets of 12 repetitions per side, maintain neutral spine
9. Wall Sits - 3 sets, hold for 30 seconds each

**Evening Session (7:00 PM - 7:30 PM):**
10. Child's Pose - 3 minutes, focus on deep breathing
11. Supine Hamstring Stretch - 45 seconds each leg, 4 repetitions
12. Bridge Exercise - 4 sets of 15 repetitions, hold top position for 2 seconds
13. Cobra Stretch - 8 repetitions, hold each for 20 seconds
14. Progressive muscle relaxation - 5 minutes

**IMMERSIVE MOTIVATIONAL NARRATIVE WITH MUSIC CUES:**

[MUSIC CUE: Play "Eye of the Tiger" by Survivor]

Good morning, John. I want you to take a deep breath right now, right where you are. Feel the air filling your lungs. That breath you just took? That's the beginning of something powerful. Two years ago, when this pain first whispered into your life, you probably never imagined it would become this constant companion, this unwelcome shadow that follows you from your desk to your dinner table, from your workspace to the moments you should be spending fully present with your children. But here's what I need you to understand, John - and I mean really understand, deep in your core - you are not defined by this pain. You are defined by what you're about to do about it.

I've seen your story before, John, but that doesn't make it any less real or any less important. Forty-five years old, sitting at that desk for ten, sometimes twelve hours, watching the clock tick while your back screams for movement, for change, for relief. You've tried the quick fixes - the massage here, the pain pill there - like putting band-aids on a dam that's starting to crack. But today is different. Today, you're not looking for temporary relief. Today, you're building a foundation for permanent change.

[MUSIC CUE: Play "Stronger" by Kanye West]

Let me tell you what I see when I look at your case, John. I see a father who wants to chase his six-year-old around the backyard without that sharp reminder in his lower back. I see a husband whose wife has noticed the way he winces when he gets up from the couch, the way he's stopped reaching for things on high shelves, the way he's slowly been withdrawing from physical life. I see someone who's gained fifteen pounds not because of lack of discipline, but because pain has made movement the enemy instead of the friend it should be. And most importantly, I see someone who's ready to fight back.

Your body is an incredible machine, John, designed for movement, for resilience, for adaptation. Right now, your core muscles have been on vacation for two years, letting your spine carry burdens it was never meant to shoulder alone. Your hip flexors have tightened from all those hours of sitting, pulling on your pelvis, tilting your spine forward, creating that deep burning sensation you know too well. Your hamstrings, tight and resistant, are pulling everything out of alignment. But here's the beautiful truth - every single one of these issues is reversible. Every single adaptation your body made to accommodate your lifestyle can be re-adapted, re-trained, rebuilt.

When you do that Cat-Cow Stretch this morning, I want you to think about what's really happening. You're not just moving your spine - you're waking up neural pathways that have been dormant. You're lubricating joints that have been stuck. You're telling your body, in the only language it truly understands - movement - that things are about to change. Each pelvic tilt is a conversation between your brain and your core muscles, reconnecting circuits that have been on standby.

[MUSIC CUE: Play "Don't Stop Believin'" by Journey]

Now, I'm not going to lie to you, John. The first week is going to test you. You're going to wake up on day three, and your back might actually feel worse before it feels better. That's not failure - that's your body waking up, that's dormant muscles learning to work again, that's adaptation in progress. You're going to have a day, probably around day five or six, where you'll think about skipping your afternoon session because work is busy, because you're tired, because the pain hasn't magically disappeared yet. And on that day, I want you to remember something: your six-year-old doesn't care about your deadlines. Your eight-year-old doesn't understand why daddy can't play soccer in the backyard. They just know they miss you. And you miss being fully present with them.

This program I've designed for you isn't about thirty minutes of exercise. It's about reclaiming your life in thirty-minute increments. Every morning when you do those Cat-Cow stretches, you're not just stretching - you're creating a ritual of self-care. Every afternoon when you take that twenty-minute break for your seated spinal twists and bird-dogs, you're not interrupting your work - you're investing in your productivity, because a body in pain is a mind distracted. Every evening when you sink into that Child's Pose, you're not just cooling down - you're practicing being present, being mindful, being connected to your body instead of at war with it.

[MUSIC CUE: Play "Hall of Fame" by The Script]

I want to talk about what success looks like, John, because I think you might be imagining this moment three months from now where suddenly you wake up pain-free, jumping out of bed like you're twenty-five again. But that's not how healing works. Success is going to be subtle at first. Week two, you'll notice you can sit for forty-five minutes instead of thirty before the pain starts. Week three, you'll realize you played with your kids for fifteen minutes without thinking about your back. Week four, your wife will comment that you seem more energetic. Week six, you'll reach for something on a high shelf without that split-second calculation of whether it's worth the pain. These small victories are the real victories, John. These are the moments that add up to transformation.

Your Bird-Dog exercises might seem simple, almost too simple to matter. But every time you extend that arm and opposite leg, holding your spine perfectly neutral, you're training the exact muscles that have been failing you. You're teaching your transverse abdominis to engage, your multifidus to stabilize, your erector spinae to support without overcompensating. These aren't just exercises - they're re-education. You're going back to school, and your body is the student.

[MUSIC CUE: Play "Lose Yourself" by Eminem]

Let me paint you a picture of where you're headed if you commit to this, John. Three months from now, you're sitting at your desk, and you realize you've been working for two hours and you haven't thought about your back once. Six months from now, you're playing in the backyard, and your son asks you to pick him up, and you do it without hesitation, without that mental calculation, without fear. Nine months from now, you've lost those fifteen pounds because movement isn't the enemy anymore - it's your ally. A year from now, your wife tells you she has her husband back, not the version of you that's been managing pain, but the version that's fully alive.

But all of that starts with today. With this morning's Cat-Cow stretches. With today's pelvic tilts. With choosing, every single day, to show up for yourself the way you show up for everyone else in your life. You've been showing up for your employer for ten to twelve hours a day. You've been showing up for your kids, for your wife, for your responsibilities. Now it's time to show up for you.

Every rep counts, John. Every stretch matters. Every time you choose to do your afternoon session instead of working through it, you're voting for the life you want instead of the life you've been settling for. And I promise you this - your body is listening. Your body is ready. Your body has been waiting for you to give it the tools it needs to heal. Now you have those tools. Now you have this plan. Now all that's left is to begin.

Welcome to your recovery, John. Welcome to reclaiming your life. Let's do this together, one stretch, one rep, one day at a time. You've got this.

**SAFETY NOTES:**
- If pain exceeds 7/10 during any exercise, stop immediately and rest
- Avoid any movements that cause sharp, shooting pain
- Progress gradually - don't increase intensity or duration by more than 10% per week
- Stay hydrated and maintain proper posture during desk work
- If pain persists or worsens after 2 weeks, consult healthcare provider"""
        },
        {
            "instruction": "Create a comprehensive daily physiotherapy plan with an immersive motivational narrative that includes music cues from the patient's playlist.",
            "patient_history": """Sarah Martinez, a 32-year-old graphic designer and fitness enthusiast, suffered a rotator cuff strain two weeks ago while attempting a new personal record on overhead press at her local gym. The injury occurred during her fourth set when her left shoulder suddenly gave out mid-lift, resulting in immediate sharp pain that she describes as feeling like something "popped" or "tore" inside. Sarah has always been active, participating in CrossFit training four times per week and rock climbing on weekends. The injury has been devastating for her psychologically as well as physically, as fitness is a core part of her identity and social life. She rates her current pain at 7/10, which spikes to 9/10 when attempting to raise her arm above shoulder height or reach behind her back. Sleep has been difficult because she can't lie on her left side, and even small movements like reaching for items on a shelf or washing her hair cause significant discomfort. Sarah is frustrated and anxious, worried that this injury might prevent her from returning to her previous fitness level. She's been doing complete rest for the first week (as initially advised) but is now eager to begin active recovery. Sarah is highly motivated but needs to understand the importance of gradual progression to prevent re-injury.""",
            "spotify_playlist": "Titanium by David Guetta, Roar by Katy Perry, Fight Song by Rachel Platten, Unwritten by Natasha Bedingfield, Brave by Sara Bareilles",
            "output": """**DAILY EXERCISE SCHEDULE:**

**Morning Session (8:30 AM - 9:00 AM):**
1. Pendulum Exercises - 6 minutes (2 min each: forward/back, side-to-side, circles)
2. Wall Slides - 3 sets of 10 repetitions, very controlled
3. Passive External Rotation (using stick) - 3 sets of 12 reps
4. Ice Therapy - 15 minutes immediately after exercises
5. Shoulder Blade Squeezes - 3 sets of 15 reps, hold 5 seconds

**Midday Session (12:30 PM - 12:50 PM):**
6. Passive Range of Motion - 10 minutes gentle movement
7. Isometric Shoulder Exercises - 5 different positions, hold 10 seconds, 5 reps each
8. Table Slides - 3 sets of 10 repetitions
9. Gentle neck stretches - 4 directions, 30 seconds each

**Evening Session (6:00 PM - 6:30 PM):**
10. Resistance Band External Rotation - 3 sets of 12 reps (VERY light resistance)
11. Doorway Chest Stretch - 45 seconds each side, 3 repetitions
12. Scapular Wall Slides - 3 sets of 8 repetitions
13. Cross-Body Stretch - 40 seconds each side, 4 repetitions
14. Ice therapy - 15 minutes

**IMMERSIVE MOTIVATIONAL NARRATIVE WITH MUSIC CUES:**

[MUSIC CUE: Play "Titanium" by David Guetta]

Sarah, I need you to listen to me right now, really listen, because what I'm about to tell you might be hard to hear, but it's exactly what you need to understand. Two weeks ago, in that moment when your shoulder gave out mid-press, when you felt that pop, when that sharp pain cut through everything else - that wasn't your body betraying you. That was your body sending you a message you'd been ignoring. And I know that's not what you want to hear. I know you're sitting there thinking about all the workouts you're missing, all the PRs you were chasing, all the Friday night climbing sessions with your friends. I know you're scared. But fear is just energy, Sarah, and we're going to channel that energy into the most important training program of your life - the one where you learn to come back stronger, smarter, and more resilient than ever before.

Let me tell you something about rotator cuff injuries that most people don't understand. Your shoulder is the most mobile joint in your entire body, capable of moving in more directions than any other joint. That mobility comes at a cost - stability. Your rotator cuff isn't just four muscles; it's a complex orchestration of strength, timing, and control. For years, you've been asking it to perform without giving it the specific attention it needed. You've been building big muscles - your deltoids, your pecs, your lats - but the small stabilizers, the rotator cuff muscles themselves, they've been working overtime trying to keep up. And two weeks ago, they couldn't keep up anymore.

[MUSIC CUE: Play "Roar" by Katy Perry]

Here's what I see when I look at your case, Sarah. I see a warrior who's been knocked down for the first time. I see someone whose entire identity has been wrapped up in being strong, being capable, being the person who never quits. I see someone who's terrified that this injury means weakness, that it means failure, that it means you're not who you thought you were. But let me tell you what I really see - I see someone who's about to learn the difference between being strong and being smart. I see someone who's about to discover that true strength isn't about how much weight you can lift; it's about having the discipline to do what your body needs, not what your ego wants.

Right now, your shoulder is inflamed. Those rotator cuff muscles - specifically your supraspinatus, the one that likely took the brunt of that overhead press - are damaged. Not destroyed, not ruined, but damaged. And damaged things can heal, Sarah. But they need the right environment to heal. They need blood flow without stress, movement without strain, activation without aggravation. That's what these pendulum exercises are about. When you lean forward and let your arm hang, when you use your body to create that gentle swinging motion, you're creating movement without muscular effort. You're pumping nutrients into that joint, you're preventing adhesions from forming, you're maintaining mobility while respecting the healing process.

I know what you're thinking. You're thinking these exercises look too easy. You're thinking you could be doing more. You're thinking about your friends at the gym, about the workouts you're missing, about falling behind. But Sarah, you're not falling behind - you're learning to build a foundation that you skipped the first time around. You're learning the lesson that every elite athlete eventually learns: rest is not weakness, rehabilitation is not regression, and smart training beats hard training every single time.

[MUSIC CUE: Play "Fight Song" by Rachel Platten]

Let me walk you through what's going to happen over the next few months, because I want you to understand the journey you're on. Week one through two - that's where you are now - is about pain management and preventing further injury. The exercises feel almost insulting in their simplicity, but they're saving your shoulder from compensatory patterns that could haunt you for years. Weeks three through four, you'll start feeling like yourself again. The sharp pain will soften into an ache, and you'll be tempted to push harder. Don't. That's the most dangerous phase because you feel better but you're not healed. Weeks five through eight, this is where we start rebuilding strength, but with intention, with control, with perfect form. Weeks nine through twelve, you'll start recognizing yourself again, but a better version - stronger where it counts, smarter about how you train, more connected to what your body is telling you.

Those isometric exercises I've prescribed? They might seem boring, pointless even. You're just pushing against a wall, not moving, not lifting, not seeing any visible progress. But inside your shoulder, inside those damaged muscle fibers, incredible things are happening. Isometric contractions allow you to generate force without moving through a painful range of motion. They're teaching your rotator cuff muscles to fire again, to engage, to stabilize, without the risk of re-injury. They're the foundation of everything that comes next.

[MUSIC CUE: Play "Unwritten" by Natasha Bedingfield]

Sarah, I want you to think about your shoulder like a book, and right now you're on a chapter you didn't want to read. The chapter about injury, about limitation, about patience. But this chapter isn't the end of your story - it's the part where the hero faces adversity and discovers what they're really made of. Every athlete, every truly successful person in any field, has a chapter like this. The ones who become great are the ones who learn from it instead of just pushing through it. You've spent years learning how to push through pain, through fatigue, through doubt. Now you're going to learn something even more valuable - how to listen.

When you do those wall slides, I want you to feel every millimeter of movement. I want you to notice where your shoulder wants to cheat, where other muscles try to compensate, where your scapula wants to wing out instead of staying controlled. This isn't just physical therapy, Sarah - this is movement education. You're learning to move the way you should have been moving all along. You're going back to basics not because you're weak, but because you're wise enough to know that basics done perfectly beat advanced movements done poorly every single time.

[MUSIC CUE: Play "Brave" by Sara Bareilles]

Let me tell you what's going to happen six months from now if you commit to this process, if you resist the urge to rush, if you embrace this journey instead of fighting it. Six months from now, you'll be back at that gym. You'll approach the overhead press again, and this time you'll notice things you never noticed before. You'll notice your scapula setting properly before you even touch the bar. You'll notice your rotator cuff muscles engaging to stabilize before your deltoids start to press. You'll notice that you're lifting with control, with intention, with awareness. And you might not lift as heavy as you did before - at least not right away - but you'll lift safer, smarter, with a foundation that will carry you through decades of training instead of leading to another injury.

But more than that, you're going to discover something about yourself. You're going to discover that you're not just strong in your body - you're strong in your mind. You're going to prove to yourself that you can be disciplined when discipline is hard, patient when patience is tested, committed when commitment isn't exciting. You're going to learn that taking two steps back to take ten steps forward isn't failure - it's strategy.

Every time you feel frustrated with these "easy" exercises, I want you to remember something: the strongest climbers aren't the ones with the most grip strength; they're the ones who know exactly when to hold on and when to let go. The best athletes aren't the ones who train the hardest; they're the ones who train the smartest. And the people who recover from injuries like yours and come back better? They're the ones who surrender the ego and embrace the process.

This is your process now, Sarah. These pendulum swings, these wall slides, these gentle stretches - they're not keeping you from training. They ARE your training. This is the most important training block of your life, even though it doesn't feel like it. You're not building muscle right now - you're building resilience. You're not setting PRs - you're setting the foundation for thousands of healthy workouts in your future.

Two weeks from now, you'll have more range of motion. Four weeks from now, you'll sleep through the night on your left side again. Six weeks from now, you'll wash your hair without thinking about it. Eight weeks from now, you'll reach for something on a high shelf and realize you didn't even hesitate. These small moments of normalcy? They're going to feel like victories, because they are victories.

Welcome to your comeback story, Sarah. It starts today, with these simple movements, with this commitment to patience, with this understanding that healing isn't linear but it is inevitable if you give your body what it needs. You didn't get injured because you're weak. You got injured because you're strong enough to push your body to its limits. Now you're going to get even stronger by learning where those limits are and how to respect them. Let's begin.

**SAFETY NOTES:**
- CRITICAL: No overhead movements above 90 degrees for first 4 weeks
- If sharp pain occurs, stop immediately - dull ache is okay, sharp is not
- Ice after every session to manage inflammation
- No CrossFit or climbing for minimum 8 weeks
- Resistance bands should be lightest possible - focus on control, not resistance"""
        },
    ]
    
    with open('physio_data_enhanced.json', 'w', encoding='utf-8') as f:
        json.dump(training_data, f, indent=2)
    
    print(f"✓ Created {len(training_data)} enhanced training examples")
    print("  - Long patient histories")
    print("  - Extended motivational narratives")
    print("  - Integrated music cues")
    return training_data

# ============================================================================
# REST OF THE CODE (SAME AS BEFORE)
# ============================================================================

def load_model_and_tokenizer():
    """Load model and tokenizer"""
    print("\nLoading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32 if USE_CPU_ONLY else torch.float16,
        device_map="cpu" if USE_CPU_ONLY else "auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    print("✓ Model loaded")
    return model, tokenizer

def setup_lora(model):
    """Configure LoRA"""
    print("\nConfiguring LoRA...")
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} ({100*trainable/total:.2f}%)")
    return model

def prepare_dataset(tokenizer):
    """Prepare dataset"""
    print("\nPreparing dataset...")
    if not os.path.exists('physio_data_enhanced.json'):
        data = create_training_data()
    else:
        with open('physio_data_enhanced.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
    
    def format_prompt(example):
        return f"""<|system|>
You are an expert physiotherapist who creates comprehensive treatment plans with immersive motivational narratives that include music cues from the patient's playlist.</s>
<|user|>
{example['instruction']}

Patient History: {example['patient_history']}

Spotify Playlist: {example['spotify_playlist']}</s>
<|assistant|>
{example['output']}</s>"""
    
    formatted_texts = [format_prompt(ex) for ex in data]
    print(f"Tokenizing {len(formatted_texts)} examples...")
    tokenized = tokenizer(
        formatted_texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )
    
    dataset = Dataset.from_dict({
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': tokenized['input_ids'].clone()
    })
    print(f"✓ Dataset ready: {len(dataset)} examples")
    return dataset

def train_model(model, tokenizer, dataset):
    """Train the model"""
    print("\nSetting up training...")
    training_args = TrainingArguments(
        output_dir="./llama-physio-checkpoints",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        save_steps=10,
        save_total_limit=2,
        fp16=False,
        report_to="none",
        max_steps=MAX_STEPS,
        warmup_steps=5,
    )
    
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=data_collator,
    )
    
    print("\n" + "="*60)
    print("Starting training...")
    print("="*60 + "\n")
    trainer.train()
    print("\n" + "="*60)
    print("Training complete!")
    print("="*60)
    return trainer

def save_model(model, tokenizer, output_dir="./llama-physio-final"):
    """Save model"""
    print(f"\nSaving model to {output_dir}...")
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✓ Model saved to {output_dir}")

def main():
    """Main pipeline"""
    try:
        model, tokenizer = load_model_and_tokenizer()
        model = setup_lora(model)
        dataset = prepare_dataset(tokenizer)
        trainer = train_model(model, tokenizer, dataset)
        save_model(model, tokenizer)
        print("\n✓ FINE-TUNING COMPLETED!")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Enhanced Physiotherapy Fine-tuning
Long Narratives + Music Integration
Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Max length: 768 tokens

Loading tokenizer...
Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


✓ Model loaded

Configuring LoRA...


The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainable: 2,252,800 (0.20%)

Preparing dataset...
Tokenizing 2 examples...
✓ Dataset ready: 2 examples

Setting up training...

Starting training...



Step,Training Loss
1,1.504200
2,1.768400
3,1.499200
4,1.752300
5,1.470700
6,1.708900
7,1.673600
8,1.382800
9,1.600700
10,1.321100


Repo card metadata block was not found. Setting CardData to empty.



Training complete!

Saving model to ./llama-physio-final...
✓ Model saved to ./llama-physio-final

✓ FINE-TUNING COMPLETED!


In [3]:
"""
Test Fine-tuned Llama Physiotherapy Model
Tests the model with various patient scenarios and evaluates outputs

Requirements:
pip install transformers peft torch
"""

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

MODEL_PATH = "./llama-physio-final"  # Path to your fine-tuned model
BASE_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Change if you used different base
USE_CPU_ONLY = True
MAX_NEW_TOKENS = 1500
TEMPERATURE = 0.7
TOP_P = 0.9

print("="*80)
print("PHYSIOTHERAPY MODEL TESTING SUITE")
print("="*80)
print(f"Model path: {MODEL_PATH}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print(f"Temperature: {TEMPERATURE}")
print("="*80 + "\n")

# ============================================================================
# TEST CASES
# ============================================================================

TEST_CASES = [
    {
        "name": "Test Case 1: Knee Pain After Running",
        "instruction": "Create a comprehensive daily physiotherapy plan with an immersive motivational narrative that includes music cues from the patient's playlist.",
        "patient_history": """Emma Johnson is a 28-year-old marathon runner who developed right knee pain three weeks ago during training. The pain is located around the kneecap and worsens during downhill running and stair climbing. She describes it as a dull, grinding sensation that rates 5/10 at rest but increases to 8/10 during activity. Emma has been training for her first Boston Marathon and is devastated about potentially missing it. She's been icing the knee and taking anti-inflammatory medication with minimal relief. Previous athletic history includes collegiate track and field. She's highly motivated but anxious about long-term damage.""",
        "spotify_playlist": "Run the World by Beyoncé, Champions by Kanye West, Can't Hold Us by Macklemore, Remember the Name by Fort Minor"
    },
]

# ============================================================================
# MODEL LOADING
# ============================================================================

def load_model():
    """Load the fine-tuned model and tokenizer"""
    print("Loading model and tokenizer...")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"  # For inference
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float32 if USE_CPU_ONLY else torch.float16,
            device_map="cpu" if USE_CPU_ONLY else "auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )
        
        model.eval()  # Set to evaluation mode
        
        print("✓ Model and tokenizer loaded successfully\n")
        return model, tokenizer
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("\nTrying to load as PEFT model...")
        
        try:
            # Try loading as PEFT model
            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            tokenizer.padding_side = "left"
            
            base_model = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL_NAME,
                torch_dtype=torch.float32 if USE_CPU_ONLY else torch.float16,
                device_map="cpu" if USE_CPU_ONLY else "auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True
            )
            
            model = PeftModel.from_pretrained(base_model, MODEL_PATH)
            model.eval()
            
            print("✓ PEFT model loaded successfully\n")
            return model, tokenizer
            
        except Exception as e2:
            print(f"❌ Error loading PEFT model: {e2}")
            raise

# ============================================================================
# INFERENCE FUNCTIONS
# ============================================================================

def format_prompt(instruction, patient_history, spotify_playlist):
    """Format the prompt for inference"""
    return f"""<|system|>
You are an expert physiotherapist who creates comprehensive treatment plans with immersive motivational narratives that include music cues from the patient's playlist.</s>
<|user|>
{instruction}

Patient History: {patient_history}

Spotify Playlist: {spotify_playlist}</s>
<|assistant|>
"""

def generate_response(model, tokenizer, prompt, max_tokens=1500):
    """Generate response from the model"""
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    
    if USE_CPU_ONLY:
        inputs = {k: v.to("cpu") for k, v in inputs.items()}
    else:
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1].strip()
    
    return response

# ============================================================================
# EVALUATION FUNCTIONS
# ============================================================================

def evaluate_response(response, test_case_name):
    """Evaluate the quality of the generated response"""
    print(f"\n{'='*80}")
    print(f"EVALUATION: {test_case_name}")
    print('='*80)
    
    # Check for key components
    checks = {
        "Has Exercise Schedule": "DAILY EXERCISE SCHEDULE" in response or "Morning Session" in response,
        "Has Motivational Narrative": len(response) > 1000,
        "Contains Music Cues": "[MUSIC CUE:" in response or "MUSIC CUE:" in response,
        "Has Safety Notes": "SAFETY" in response or "safety" in response.lower(),
        "Includes Specific Exercises": any(ex in response.lower() for ex in ["stretch", "exercise", "repetition", "sets"]),
        "Mentions Patient by Name": True,  # Would need to extract name from history
        "Has Time-based Structure": any(time in response for time in ["AM", "PM", "morning", "afternoon", "evening"]),
        "Appropriate Length": 800 < len(response) < 3000,
    }
    
    print("\nQuality Checks:")
    passed = 0
    for check, result in checks.items():
        status = "✓" if result else "✗"
        print(f"  {status} {check}")
        if result:
            passed += 1
    
    score = (passed / len(checks)) * 100
    print(f"\nOverall Score: {score:.1f}% ({passed}/{len(checks)} checks passed)")
    
    # Word count and structure analysis
    words = len(response.split())
    music_cues = response.count("[MUSIC CUE:") + response.count("MUSIC CUE:")
    
    print(f"\nResponse Statistics:")
    print(f"  - Total words: {words:,}")
    print(f"  - Total characters: {len(response):,}")
    print(f"  - Music cues found: {music_cues}")
    print(f"  - Estimated reading time: {words // 200} minutes")
    
    return score

# ============================================================================
# MAIN TEST FUNCTION
# ============================================================================

def run_tests():
    """Run all test cases"""
    print("Starting test suite...\n")
    
    # Load model
    model, tokenizer = load_model()
    
    results = []
    
    # Run each test case
    for i, test_case in enumerate(TEST_CASES, 1):
        print("\n" + "="*80)
        print(f"RUNNING TEST {i}/{len(TEST_CASES)}: {test_case['name']}")
        print("="*80)
        
        # Format prompt
        prompt = format_prompt(
            test_case['instruction'],
            test_case['patient_history'],
            test_case['spotify_playlist']
        )
        
        print("\nGenerating response...")
        print("(This may take a few minutes...)\n")
        
        # Generate response
        response = generate_response(model, tokenizer, prompt, max_tokens=MAX_NEW_TOKENS)
        
        # Save response to file
        output_file = f"test_output_{i}.txt"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(f"TEST CASE: {test_case['name']}\n")
            f.write("="*80 + "\n\n")
            f.write("PATIENT HISTORY:\n")
            f.write(test_case['patient_history'] + "\n\n")
            f.write("SPOTIFY PLAYLIST:\n")
            f.write(test_case['spotify_playlist'] + "\n\n")
            f.write("="*80 + "\n")
            f.write("MODEL RESPONSE:\n")
            f.write("="*80 + "\n\n")
            f.write(response)
        
        print(f"✓ Response saved to {output_file}")
        
        # Evaluate response
        score = evaluate_response(response, test_case['name'])
        
        results.append({
            'test_case': test_case['name'],
            'score': score,
            'output_file': output_file
        })
        
        # Print preview of response
        print("\n" + "-"*80)
        print("RESPONSE PREVIEW (first 500 characters):")
        print("-"*80)
        print(response[:500] + "..." if len(response) > 500 else response)
        print("-"*80)
    
    # Summary
    print("\n\n" + "="*80)
    print("TEST SUITE SUMMARY")
    print("="*80)
    
    for result in results:
        print(f"\n{result['test_case']}")
        print(f"  Score: {result['score']:.1f}%")
        print(f"  Output: {result['output_file']}")
    
    avg_score = sum(r['score'] for r in results) / len(results)
    print(f"\n{'='*80}")
    print(f"AVERAGE SCORE: {avg_score:.1f}%")
    print("="*80)
    
    # Save summary
    with open('test_summary.txt', 'w', encoding='utf-8') as f:
        f.write("PHYSIOTHERAPY MODEL TEST SUMMARY\n")
        f.write("="*80 + "\n\n")
        f.write(f"Model Path: {MODEL_PATH}\n")
        f.write(f"Test Cases: {len(TEST_CASES)}\n")
        f.write(f"Average Score: {avg_score:.1f}%\n\n")
        f.write("="*80 + "\n")
        f.write("INDIVIDUAL RESULTS\n")
        f.write("="*80 + "\n\n")
        for result in results:
            f.write(f"{result['test_case']}: {result['score']:.1f}%\n")
    
    print("\n✓ Summary saved to test_summary.txt")
    print("\nTesting complete!")

# ============================================================================
# INTERACTIVE MODE
# ============================================================================

def interactive_mode():
    """Interactive testing mode"""
    print("\n" + "="*80)
    print("INTERACTIVE MODE")
    print("="*80)
    print("Test the model with your own patient scenarios!")
    print("Type 'quit' to exit.\n")
    
    model, tokenizer = load_model()
    
    while True:
        print("\n" + "-"*80)
        instruction = input("Instruction (or 'quit'): ").strip()
        if instruction.lower() == 'quit':
            break
        
        if not instruction:
            instruction = "Create a comprehensive daily physiotherapy plan with an immersive motivational narrative that includes music cues from the patient's playlist."
        
        print()
        patient_history = input("Patient History: ").strip()
        if not patient_history:
            print("Patient history is required!")
            continue
        
        print()
        playlist = input("Spotify Playlist (optional): ").strip()
        if not playlist:
            playlist = "Eye of the Tiger by Survivor, Stronger by Kanye West"
        
        prompt = format_prompt(instruction, patient_history, playlist)
        
        print("\n" + "="*80)
        print("GENERATING RESPONSE...")
        print("="*80 + "\n")
        
        response = generate_response(model, tokenizer, prompt, max_tokens=MAX_NEW_TOKENS)
        
        print(response)
        print("\n" + "="*80)

# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main function"""
    import sys
    
    if len(sys.argv) > 1 and sys.argv[1] == '--interactive':
        interactive_mode()
    else:
        run_tests()

if __name__ == "__main__":
    main()

PHYSIOTHERAPY MODEL TESTING SUITE
Model path: ./llama-physio-final
Max new tokens: 1500
Temperature: 0.7

Starting test suite...

Loading model and tokenizer...
✓ Model and tokenizer loaded successfully


RUNNING TEST 1/1: Test Case 1: Knee Pain After Running

Generating response...
(This may take a few minutes...)

✓ Response saved to test_output_1.txt

EVALUATION: Test Case 1: Knee Pain After Running

Quality Checks:
  ✓ Has Exercise Schedule
  ✓ Has Motivational Narrative
  ✗ Contains Music Cues
  ✗ Has Safety Notes
  ✓ Includes Specific Exercises
  ✓ Mentions Patient by Name
  ✓ Has Time-based Structure
  ✗ Appropriate Length

Overall Score: 62.5% (5/8 checks passed)

Response Statistics:
  - Total words: 829
  - Total characters: 5,092
  - Music cues found: 0
  - Estimated reading time: 4 minutes

--------------------------------------------------------------------------------
RESPONSE PREVIEW (first 500 characters):
----------------------------------------------------------------

In [2]:
"""
Quick Test Script for Fine-tuned Physiotherapy Model
Fast validation with minimal setup

Requirements:
pip install transformers peft torch
"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

MODEL_PATH = "./llama-physio-final"
BASE_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
USE_CPU_ONLY = True

print("="*80)
print("QUICK MODEL TEST")
print("="*80 + "\n")

# Simple test case
QUICK_TEST = {
    "patient_history": "Mark, 40, has lower back pain from sitting at desk job. Pain level 6/10. Wants quick relief.",
    "playlist": "Eye of the Tiger, Stronger"
}

# ============================================================================
# FUNCTIONS
# ============================================================================

def load_model():
    """Load model quickly"""
    print("Loading model...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.float32 if USE_CPU_ONLY else torch.float16,
            device_map="cpu" if USE_CPU_ONLY else "auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )
        model.eval()
        print("✓ Model loaded\n")
        return model, tokenizer
    except:
        print("Loading as PEFT...")
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=torch.float32 if USE_CPU_ONLY else torch.float16,
            device_map="cpu" if USE_CPU_ONLY else "auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )
        model = PeftModel.from_pretrained(base, MODEL_PATH)
        model.eval()
        print("✓ PEFT model loaded\n")
        return model, tokenizer

def test_model():
    """Quick test"""
    model, tokenizer = load_model()
    
    prompt = f"""<|system|>
You are an expert physiotherapist who creates comprehensive treatment plans with immersive motivational narratives that include music cues from the patient's playlist.</s>
<|user|>
Create a comprehensive daily physiotherapy plan with an immersive motivational narrative that includes music cues from the patient's playlist.

Patient History: {QUICK_TEST['patient_history']}

Spotify Playlist: {QUICK_TEST['playlist']}</s>
<|assistant|>
"""
    
    print("Generating response...\n")
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    
    if USE_CPU_ONLY:
        inputs = {k: v.to("cpu") for k, v in inputs.items()}
    else:
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1].strip()
    
    print("="*80)
    print("RESPONSE:")
    print("="*80 + "\n")
    print(response)
    print("\n" + "="*80)
    
    # Quick checks
    print("\nQUICK CHECKS:")
    checks = [
        ("Has exercises", any(word in response.lower() for word in ["exercise", "stretch", "sets", "repetitions"])),
        ("Has structure", any(word in response.lower() for word in ["session", "morning", "schedule"])),
        ("Has motivation", len(response) > 300),
        ("Has music cues", "[MUSIC CUE:" in response or "music" in response.lower()),
        ("Has safety notes", "safety" in response.lower() or "pain" in response.lower()),
    ]
    
    passed = 0
    for check, result in checks:
        print(f"  {'✓' if result else '✗'} {check}")
        if result:
            passed += 1
    
    print(f"\nScore: {passed}/{len(checks)} ({100*passed/len(checks):.0f}%)")
    print("\n✓ Quick test complete!")
    
    # Save
    with open('quick_test_output.txt', 'w', encoding='utf-8') as f:
        f.write("QUICK TEST OUTPUT\n")
        f.write("="*80 + "\n\n")
        f.write(response)
    print("✓ Output saved to quick_test_output.txt")

if __name__ == "__main__":
    try:
        test_model()
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

QUICK MODEL TEST

Loading model...
✓ Model loaded

Generating response...



KeyboardInterrupt: 

In [ ]:
"""
Physiotherapy Audio Generation System
Generates motivational narratives with integrated music cues

Requirements:
pip install transformers peft torch gtts pydub
"""

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import re
import os

# For TTS and audio mixing (optional - install if you want full audio generation)
try:
    from gtts import gTTS
    from pydub import AudioSegment
    from pydub.playback import play
    AUDIO_AVAILABLE = True
except ImportError:
    print("⚠️  Audio libraries not installed. Install with: pip install gtts pydub")
    AUDIO_AVAILABLE = False

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FINETUNED_PATH = "./llama-physio-final"
USE_CPU = True

# ============================================================================
# LOAD MODEL
# ============================================================================

class PhysioNarrativeGenerator:
    """Generate physiotherapy narratives with music integration"""
    
    def __init__(self):
        print("Loading fine-tuned model...")
        
        self.base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            torch_dtype=torch.float32 if USE_CPU else torch.float16,
            device_map="cpu" if USE_CPU else "auto",
            low_cpu_mem_usage=True
        )
        
        self.model = PeftModel.from_pretrained(self.base_model, FINETUNED_PATH)
        self.model.eval()
        self.tokenizer = AutoTokenizer.from_pretrained(FINETUNED_PATH)
        
        print("✓ Model loaded!\n")
    
    def generate_plan(self, patient_history, spotify_playlist):
        """Generate comprehensive plan with long narrative and music cues"""
        
        prompt = f"""<|system|>
You are an expert physiotherapist who creates comprehensive treatment plans with immersive motivational narratives that include music cues from the patient's playlist.</s>
<|user|>
Create a comprehensive daily physiotherapy plan with an immersive motivational narrative that includes music cues from the patient's playlist.

Patient History: {patient_history}

Spotify Playlist: {spotify_playlist}</s>
<|assistant|>"""
        
        inputs = self.tokenizer(prompt, return_tensors="pt")
        if not USE_CPU:
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        print("Generating plan... (this may take 1-2 minutes)")
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=1500,  # Increased for long narratives
                temperature=0.8,
                top_p=0.95,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        if '<|assistant|>' in response:
            response = response.split('<|assistant|>')[-1].strip()
        
        return response
    
    def parse_music_cues(self, narrative):
        """Extract music cues from narrative"""
        # Pattern: [MUSIC CUE: Play "Song" by Artist]
        pattern = r'\[MUSIC CUE: Play "([^"]+)" by ([^\]]+)\]'
        cues = re.findall(pattern, narrative)
        
        music_cues = []
        for song, artist in cues:
            music_cues.append({
                'song': song,
                'artist': artist,
                'full': f"{song} by {artist}"
            })
        
        return music_cues
    
    def split_narrative_segments(self, narrative):
        """Split narrative into segments between music cues"""
        # Split by music cue markers
        pattern = r'\[MUSIC CUE: Play "[^""]+" by [^\]]+\]'
        segments = re.split(pattern, narrative)
        
        # Get music cues
        music_cues = self.parse_music_cues(narrative)
        
        # Combine segments with music cues
        result = []
        for i, segment in enumerate(segments):
            segment = segment.strip()
            if segment:
                result.append({
                    'type': 'narration',
                    'content': segment
                })
            if i < len(music_cues):
                result.append({
                    'type': 'music',
                    'content': music_cues[i]['full'],
                    'song': music_cues[i]['song'],
                    'artist': music_cues[i]['artist']
                })
        
        return result
    
    def generate_audio(self, narrative, output_dir="./audio_output"):
        """Generate audio files from narrative (requires gtts and pydub)"""
        
        if not AUDIO_AVAILABLE:
            print("⚠️  Audio generation unavailable. Install: pip install gtts pydub")
            return None
        
        os.makedirs(output_dir, exist_ok=True)
        
        segments = self.split_narrative_segments(narrative)
        audio_segments = []
        
        print("\nGenerating audio segments...")
        
        for i, segment in enumerate(segments):
            if segment['type'] == 'narration':
                print(f"  [{i+1}/{len(segments)}] Narration segment ({len(segment['content'])} chars)")
                
                # Generate TTS
                tts = gTTS(text=segment['content'], lang='en', slow=False)
                filename = f"{output_dir}/segment_{i:03d}_narration.mp3"
                tts.save(filename)
                
                audio_segments.append({
                    'file': filename,
                    'type': 'narration'
                })
                
            elif segment['type'] == 'music':
                print(f"  [{i+1}/{len(segments)}] Music cue: {segment['content']}")
                
                # Note: You would need actual music files for this
                # For now, we'll create a placeholder
                audio_segments.append({
                    'type': 'music_cue',
                    'song': segment['song'],
                    'artist': segment['artist'],
                    'note': 'Insert actual song file here'
                })
        
        print(f"\n✓ Generated {len(audio_segments)} audio segments")
        print(f"  Saved to: {output_dir}/")
        
        return audio_segments

# ============================================================================
# EXAMPLE USAGE
# ============================================================================

def main():
    # Initialize generator
    generator = PhysioNarrativeGenerator()
    
    # Example patient data
    patient_history = """Michael Thompson is a 50-year-old construction worker who has been experiencing severe lower back pain for the past six months. The pain started after lifting heavy materials without proper form on a job site. He describes the pain as a constant dull ache that intensifies to sharp, shooting pain when he bends forward or twists his torso. Michael rates his pain at 7/10 on average, reaching 9/10 during flare-ups. He has been taking ibuprofen regularly, which provides minimal relief. Michael is worried about his ability to continue working and supporting his family. He has three children in school and cannot afford to take extended time off work. Previous attempts at recovery included rest and over-the-counter pain medication, but the pain always returns when he goes back to work. Michael is highly motivated to recover but needs education on proper body mechanics and a sustainable exercise program that he can maintain alongside his physically demanding job. He works 10-hour shifts, often in awkward positions, and has limited time for lengthy workout routines."""
    
    spotify_playlist = "Eye of the Tiger by Survivor, Lose Yourself by Eminem, Stronger by Kanye West, Till I Collapse by Eminem, Remember the Name by Fort Minor"
    
    print("="*60)
    print("ENHANCED PHYSIOTHERAPY NARRATIVE GENERATION")
    print("="*60)
    print("\nPatient History:")
    print(patient_history[:200] + "...")
    print("\nSpotify Playlist:")
    print(spotify_playlist)
    print("\n" + "="*60 + "\n")
    
    # Generate plan
    plan = generator.generate_plan(patient_history, spotify_playlist)
    
    # Display results
    print("\n" + "="*60)
    print("GENERATED PLAN")
    print("="*60 + "\n")
    print(plan)
    
    # Parse music cues
    print("\n" + "="*60)
    print("MUSIC INTEGRATION ANALYSIS")
    print("="*60)
    
    music_cues = generator.parse_music_cues(plan)
    print(f"\nFound {len(music_cues)} music cues in narrative:")
    for i, cue in enumerate(music_cues, 1):
        print(f"  {i}. {cue['full']}")
    
    segments = generator.split_narrative_segments(plan)
    print(f"\nNarrative structure:")
    print(f"  - {sum(1 for s in segments if s['type'] == 'narration')} narration segments")
    print(f"  - {sum(1 for s in segments if s['type'] == 'music')} music cues")
    
    # Optional: Generate audio
    if AUDIO_AVAILABLE:
        print("\n" + "="*60)
        print("AUDIO GENERATION")
        print("="*60)
        
        response = input("\nGenerate audio files? (y/n): ")
        if response.lower() == 'y':
            audio_segments = generator.generate_audio(plan)
    else:
        print("\n💡 TIP: Install audio libraries to generate speech:")
        print("   pip install gtts pydub")
    
    # Save to file
    output_file = "physio_plan_output.txt"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("PHYSIOTHERAPY PLAN WITH MUSIC INTEGRATION\n")
        f.write("="*60 + "\n\n")
        f.write(f"Patient History:\n{patient_history}\n\n")
        f.write(f"Spotify Playlist:\n{spotify_playlist}\n\n")
        f.write("="*60 + "\n")
        f.write("GENERATED PLAN\n")
        f.write("="*60 + "\n\n")
        f.write(plan)
    
    print(f"\n✓ Plan saved to: {output_file}")

if __name__ == "__main__":
    main()

"""
COMPLETE WORKFLOW:

1. TRAIN THE MODEL:
   python finetune_llama.py

2. GENERATE NARRATIVES:
   python audio_generation_system.py

3. AUDIO GENERATION (Optional):
   - Install: pip install gtts pydub ffmpeg-python
   - Run the script and choose 'y' for audio generation
   - Add actual music files to mix with narration

MUSIC INTEGRATION:
- Model generates [MUSIC CUE: Play "Song" by Artist] markers
- Parser extracts these and splits narrative into segments
- TTS generates speech for each narration segment
- Music files can be mixed in at the cue points

NEXT STEPS:
- Collect actual music files from Spotify
- Use pydub to mix narration + music
- Add fade in/out effects
- Adjust volume levels (narration louder, music as background)
"""

Loading fine-tuned model...


`torch_dtype` is deprecated! Use `dtype` instead!


✓ Model loaded!

ENHANCED PHYSIOTHERAPY NARRATIVE GENERATION

Patient History:
Michael Thompson is a 50-year-old construction worker who has been experiencing severe lower back pain for the past six months. The pain started after lifting heavy materials without proper form on a ...

Spotify Playlist:
Eye of the Tiger by Survivor, Lose Yourself by Eminem, Stronger by Kanye West, Till I Collapse by Eminem, Remember the Name by Fort Minor


Generating plan... (this may take 1-2 minutes)

GENERATED PLAN

**DAILY EXERCISE SCHEDULE:**

**Morning Session (8:30 AM - 9:00 AM):**
1. Kettlebell Swings - 3 sets of 8 repetitions, 30 seconds rest between each repetition
2. Dead Bug Exercise - 4 sets of 8 repetitions per side
3. Knee-to-Chest Exercises - Include knee elevation, internal/external rotations
4. Passive Range of Motion - Spend 5 minutes each pass
5. Seated Spinal Twists - 3 sets of 15 repetitions per side
6. Ladder Bench Press - 3 sets of 12 reps, focus on form
7. Jumping Jacks - 45 seco

In [4]:
"""
Professional Audio Mixing System with Pydub
Creates high-quality single MP3 with proper mixing

Requirements:
pip install gtts pydub
conda install -c conda-forge ffmpeg

This version creates professional-quality audio!
"""
import os
import subprocess

# Find FFmpeg location
try:
    result = subprocess.run(['where', 'ffmpeg'], capture_output=True, text=True)
    ffmpeg_path = result.stdout.strip().split('\n')[0]
    ffmpeg_dir = os.path.dirname(ffmpeg_path)
    
    # Set environment variables for pydub
    os.environ["PATH"] += os.pathsep + ffmpeg_dir
    
    print(f"✓ FFmpeg found at: {ffmpeg_path}")
except:
    print("⚠️  FFmpeg not found in PATH")

# NOW import pydub
from pydub import AudioSegment
from pydub.utils import which

# Manually set ffmpeg paths
AudioSegment.converter = ffmpeg_path
AudioSegment.ffmpeg = ffmpeg_path
AudioSegment.ffprobe = ffmpeg_path.replace('ffmpeg.exe', 'ffprobe.exe')

print(f"✓ Pydub configured with FFmpeg")



import os
import re
from gtts import gTTS
from pydub import AudioSegment
from pydub.effects import normalize
import time

class ProfessionalAudioMixer:
    """Professional audio mixing with proper transitions"""
    
    def __init__(self, music_folder="./music", output_folder="./final_audio"):
        self.music_folder = music_folder
        self.output_folder = output_folder
        self.temp_folder = os.path.join(output_folder, "temp_segments")
        
        os.makedirs(output_folder, exist_ok=True)
        os.makedirs(music_folder, exist_ok=True)
        os.makedirs(self.temp_folder, exist_ok=True)
        
        print("="*60)
        print("PROFESSIONAL PHYSIOTHERAPY AUDIO MIXER")
        print("High-quality mixing with Pydub")
        print("="*60)
        
        # Check FFmpeg
        self.check_ffmpeg()
    
    def check_ffmpeg(self):
        """Check if FFmpeg is available"""
        try:
            AudioSegment.converter
            print("✓ FFmpeg detected - ready for professional mixing")
            return True
        except:
            print("\n⚠️  FFmpeg NOT detected!")
            print("   Install with: conda install -c conda-forge ffmpeg")
            print("   Then restart this script\n")
            return False
    
    def parse_narrative_with_cues(self, text):
        """Parse narrative and extract music cues"""
        pattern = r'\[MUSIC CUE: Play "([^"]+)" by ([^\]]+)\]'
        parts = []
        last_end = 0
        
        for match in re.finditer(pattern, text):
            if match.start() > last_end:
                narration = text[last_end:match.start()].strip()
                if narration:
                    parts.append({'type': 'narration', 'content': narration})
            
            parts.append({
                'type': 'music',
                'song': match.group(1),
                'artist': match.group(2),
                'duration': 15  # seconds
            })
            last_end = match.end()
        
        if last_end < len(text):
            remaining = text[last_end:].strip()
            if remaining:
                parts.append({'type': 'narration', 'content': remaining})
        
        return parts
    
    def list_available_music(self):
        """List all music files in the music folder"""
        if not os.path.exists(self.music_folder):
            return {}
        
        music_files = {}
        for file in os.listdir(self.music_folder):
            if file.endswith(('.mp3', '.wav', '.m4a', '.flac')):
                name = os.path.splitext(file)[0]
                music_files[name.lower()] = os.path.join(self.music_folder, file)
        
        return music_files
    
    def find_music_file(self, song, artist):
        """Find music file with fuzzy matching"""
        available_music = self.list_available_music()
        
        # Try exact matches first
        search_patterns = [
            f"{song} - {artist}".lower(),
            f"{artist} - {song}".lower(),
            song.lower(),
        ]
        
        for pattern in search_patterns:
            if pattern in available_music:
                print(f"  ✓ Found: {os.path.basename(available_music[pattern])}")
                return available_music[pattern]
        
        # Partial match
        song_words = set(song.lower().split())
        for key, filepath in available_music.items():
            key_words = set(key.split())
            if len(song_words & key_words) >= 2:  # At least 2 words match
                print(f"  ✓ Found (partial): {os.path.basename(filepath)}")
                return filepath
        
        print(f"  ✗ Not found: {song} by {artist}")
        return None
    
    def generate_tts_audio(self, text, temp_file):
        """Generate TTS and return AudioSegment"""
        print(f"  Generating TTS ({len(text)} chars)...")
        
        try:
            # Generate TTS
            tts = gTTS(text=text, lang='en', slow=False)
            tts.save(temp_file)
            
            # Load as AudioSegment
            audio = AudioSegment.from_mp3(temp_file)
            
            # Normalize
            audio = normalize(audio)
            
            duration = len(audio) / 1000  # seconds
            print(f"  ✓ Generated: {duration:.1f}s")
            
            return audio
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            return None
    
    def load_music_segment(self, music_file, duration):
        """Load and prepare music segment"""
        
        try:
            print(f"  Loading music: {os.path.basename(music_file)}")
            
            # Load music
            music = AudioSegment.from_file(music_file)
            
            # Take only specified duration
            duration_ms = duration * 1000
            music_clip = music[:duration_ms]
            
            # Reduce volume for background music
            music_clip = music_clip - 12  # Reduce by 12dB
            
            # Add fade in/out
            fade_duration = min(2000, len(music_clip) // 4)  # 2 seconds or 1/4 of clip
            music_clip = music_clip.fade_in(fade_duration).fade_out(fade_duration)
            
            print(f"  ✓ Prepared: {duration}s at -12dB with fades")
            
            return music_clip
            
        except Exception as e:
            print(f"  ✗ Error loading music: {e}")
            return None
    
    def mix_audio_segments(self, parts, output_name="motivation_session"):
        """Mix all segments with professional quality"""
        
        # Check available music
        available_music = self.list_available_music()
        print(f"\n✓ Found {len(available_music)} music file(s) in {self.music_folder}/")
        if available_music:
            for filepath in list(available_music.values())[:5]:
                size = os.path.getsize(filepath) / (1024*1024)  # MB
                print(f"  - {os.path.basename(filepath)} ({size:.1f} MB)")
            if len(available_music) > 5:
                print(f"  ... and {len(available_music) - 5} more")
        else:
            print("  ⚠️  No music files found - will generate narration only")
        
        # Generate all segments
        print("\n" + "="*60)
        print("GENERATING AUDIO SEGMENTS")
        print("="*60)
        
        audio_segments = []
        segment_info = []
        
        for i, part in enumerate(parts, 1):
            print(f"\n[Segment {i}/{len(parts)}]")
            
            if part['type'] == 'narration':
                temp_file = os.path.join(self.temp_folder, f"narration_{i:03d}.mp3")
                audio = self.generate_tts_audio(part['content'], temp_file)
                
                if audio:
                    audio_segments.append(audio)
                    segment_info.append({
                        'type': 'narration',
                        'duration': len(audio) / 1000
                    })
            
            elif part['type'] == 'music':
                music_file = self.find_music_file(part['song'], part['artist'])
                
                if music_file:
                    audio = self.load_music_segment(music_file, part['duration'])
                    
                    if audio:
                        audio_segments.append(audio)
                        segment_info.append({
                            'type': 'music',
                            'song': part['song'],
                            'duration': len(audio) / 1000
                        })
                else:
                    print(f"  → Skipping this music segment")
        
        if not audio_segments:
            print("\n✗ No audio segments generated!")
            return None
        
        # Combine segments
        print("\n" + "="*60)
        print("MIXING AUDIO")
        print("="*60)
        
        print(f"\nCombining {len(audio_segments)} segments...")
        
        # Start with silence
        final_audio = AudioSegment.silent(duration=500)
        
        for i, (audio, info) in enumerate(zip(audio_segments, segment_info), 1):
            segment_type = "MUSIC" if info['type'] == 'music' else "NARRATION"
            print(f"  [{i}/{len(audio_segments)}] Adding {segment_type} ({info['duration']:.1f}s)")
            
            # Add segment
            final_audio += audio
            
            # Add pause between segments
            pause_duration = 800 if info['type'] == 'narration' else 500
            final_audio += AudioSegment.silent(duration=pause_duration)
        
        # Final silence
        final_audio += AudioSegment.silent(duration=1000)
        
        # Normalize final audio
        print("\n  Normalizing audio...")
        final_audio = normalize(final_audio)
        
        # Export
        output_file = os.path.join(self.output_folder, f"{output_name}.mp3")
        print(f"  Exporting to MP3...")
        
        final_audio.export(
            output_file, 
            format="mp3",
            bitrate="192k",
            parameters=["-q:a", "0"]  # Highest quality
        )
        
        # Create summary
        self.create_summary(segment_info, output_file)
        
        # Cleanup
        print("\n  Cleaning up temporary files...")
        import shutil
        try:
            shutil.rmtree(self.temp_folder)
            print("  ✓ Cleanup complete")
        except:
            pass
        
        return output_file
    
    def create_summary(self, segment_info, output_file):
        """Create session summary"""
        
        file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
        total_duration = sum(s['duration'] for s in segment_info)
        
        narration_count = sum(1 for s in segment_info if s['type'] == 'narration')
        music_count = sum(1 for s in segment_info if s['type'] == 'music')
        
        narration_duration = sum(s['duration'] for s in segment_info if s['type'] == 'narration')
        music_duration = sum(s['duration'] for s in segment_info if s['type'] == 'music')
        
        print("\n" + "="*60)
        print("✓ PROFESSIONAL AUDIO COMPLETE!")
        print("="*60)
        
        print(f"\nOutput file: {os.path.basename(output_file)}")
        print(f"Location: {os.path.dirname(output_file)}")
        print(f"File size: {file_size:.1f} MB")
        print(f"Quality: 192 kbps MP3")
        
        print(f"\nTotal duration: {total_duration/60:.1f} minutes ({total_duration:.0f}s)")
        print(f"  - Narration: {narration_duration:.0f}s ({narration_count} segments)")
        print(f"  - Music: {music_duration:.0f}s ({music_count} segments)")
        
        print(f"\nSegment breakdown:")
        for i, info in enumerate(segment_info, 1):
            if info['type'] == 'music':
                print(f"  {i}. Music: {info['song']} ({info['duration']:.0f}s)")
            else:
                print(f"  {i}. Narration ({info['duration']:.0f}s)")
        
        print(f"\n✓ Ready to play in any media player!")
        print(f"✓ High-quality professional mix with smooth transitions")


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

def create_example_narrative():
    """Example narrative with music cues"""
    
    return """**DAILY EXERCISE SCHEDULE:**

Morning (7:30 AM): Cat-Cow Stretch, Pelvic Tilts, Walking
Afternoon (2:00 PM): Bird-Dog Exercise, Seated Twists
Evening (7:00 PM): Child's Pose, Hamstring Stretches

**MOTIVATIONAL NARRATIVE:**

[MUSIC CUE: Play "Eye of the Tiger" by Survivor]

Good morning, John. Today is a new chapter in your recovery journey. I know your back has been giving you trouble, but I want you to understand something important - every single rep you do today is building toward a pain-free future. Two years of sitting at that desk has created patterns in your body, but patterns can be rewritten. You're not just doing exercises, John. You're retraining your body, reconnecting with muscles that have been dormant, rebuilding the foundation of your core strength.

[MUSIC CUE: Play "Stronger" by Kanye West]

Think about your children, John. Think about playing in the backyard without that constant reminder of pain. Think about reaching for things on high shelves without hesitation. Think about sleeping through the night on any position you want. That's where we're headed. Every Cat-Cow stretch is telling your spine that movement is safe again. Every pelvic tilt is waking up those deep core muscles. Every Bird-Dog exercise is teaching your body stability and control.

[MUSIC CUE: Play "Don't Stop Believin'" by Journey]

I've seen hundreds of cases like yours, and I can tell you with absolute certainty - you can beat this. The pain you feel today is not permanent. It's a message from your body asking for change, and you're answering that call. Three months from now, you'll look back at today and realize this was the turning point. This was the day you decided to take control. Stay consistent, stay patient, and trust the process. Your body knows how to heal - you just need to give it the right environment.

Welcome to your recovery, John. Let's do this together."""


def main():
    print("="*60)
    print("PROFESSIONAL PHYSIOTHERAPY AUDIO MIXER")
    print("="*60)
    
    # Initialize mixer
    mixer = ProfessionalAudioMixer()
    
    # Get narrative
    narrative = create_example_narrative()
    
    print("\nNarrative Preview:")
    print("-"*60)
    print(narrative[:250] + "...")
    print("-"*60)
    
    # Parse
    parts = mixer.parse_narrative_with_cues(narrative)
    
    print(f"\nParsed {len(parts)} segments:")
    for i, part in enumerate(parts, 1):
        if part['type'] == 'narration':
            print(f"  {i}. Narration ({len(part['content'])} chars)")
        else:
            print(f"  {i}. Music: {part['song']} by {part['artist']}")
    
    # Mix
    output_file = mixer.mix_audio_segments(parts, "john_motivation_professional")
    
    if output_file:
        print("\n" + "="*60)
        print("SUCCESS!")
        print("="*60)
        print(f"\nPlay your professional-quality audio:")
        print(f"{output_file}")


if __name__ == "__main__":
    main()


"""
SETUP GUIDE:

1. INSTALL REQUIREMENTS:
   pip install gtts pydub
   
2. INSTALL FFMPEG (REQUIRED):
   conda install -c conda-forge ffmpeg
   
   Without FFmpeg, this won't work!

3. ADD MUSIC:
   Place MP3 files in ./music/ folder
   Any naming works - fuzzy matching enabled

4. RUN:
   python audio_mixer.py

FEATURES:
✓ Professional quality 192kbps MP3
✓ Smooth transitions with fades
✓ Music at -12dB (perfect background level)
✓ Normalized audio throughout
✓ Proper silence between segments
✓ Single combined MP3 output

RESULT:
High-quality professional audio ready for patients!
"""

✓ FFmpeg found at: 
✓ Pydub configured with FFmpeg
PROFESSIONAL PHYSIOTHERAPY AUDIO MIXER
PROFESSIONAL PHYSIOTHERAPY AUDIO MIXER
High-quality mixing with Pydub
✓ FFmpeg detected - ready for professional mixing

Narrative Preview:
------------------------------------------------------------
**DAILY EXERCISE SCHEDULE:**

Morning (7:30 AM): Cat-Cow Stretch, Pelvic Tilts, Walking
Afternoon (2:00 PM): Bird-Dog Exercise, Seated Twists
Evening (7:00 PM): Child's Pose, Hamstring Stretches

**MOTIVATIONAL NARRATIVE:**

[MUSIC CUE: Play "Eye of ...
------------------------------------------------------------

Parsed 7 segments:
  1. Narration (223 chars)
  2. Music: Eye of the Tiger by Survivor
  3. Narration (501 chars)
  4. Music: Stronger by Kanye West
  5. Narration (466 chars)
  6. Music: Don't Stop Believin' by Journey
  7. Narration (550 chars)

✓ Found 12 music file(s) in ./music/
  - David Guetta, Sia, MORTEN - Titanium (feat. Sia) - David Guetta & MORTEN Future Rave Remix.mp3 (3.4 MB)


c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)


  ✗ Error: [WinError 2] The system cannot find the file specified

[Segment 2/7]
  ✓ Found: Survivor - Eye of the Tiger.mp3
  Loading music: Survivor - Eye of the Tiger.mp3
  ✗ Error loading music: [WinError 2] The system cannot find the file specified

[Segment 3/7]
  Generating TTS (501 chars)...


c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)


  ✗ Error: [WinError 2] The system cannot find the file specified

[Segment 4/7]
  ✓ Found: Kanye West - Stronger.mp3
  Loading music: Kanye West - Stronger.mp3
  ✗ Error loading music: [WinError 2] The system cannot find the file specified

[Segment 5/7]
  Generating TTS (466 chars)...


c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)


  ✗ Error: [WinError 2] The system cannot find the file specified

[Segment 6/7]
  ✓ Found: Journey - Don't Stop Believin'.mp3
  Loading music: Journey - Don't Stop Believin'.mp3
  ✗ Error loading music: [WinError 2] The system cannot find the file specified

[Segment 7/7]
  Generating TTS (550 chars)...
  ✗ Error: [WinError 2] The system cannot find the file specified

✗ No audio segments generated!


c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)


"\nSETUP GUIDE:\n\n1. INSTALL REQUIREMENTS:\n   pip install gtts pydub\n   \n2. INSTALL FFMPEG (REQUIRED):\n   conda install -c conda-forge ffmpeg\n   \n   Without FFmpeg, this won't work!\n\n3. ADD MUSIC:\n   Place MP3 files in ./music/ folder\n   Any naming works - fuzzy matching enabled\n\n4. RUN:\n   python audio_mixer.py\n\nFEATURES:\n✓ Professional quality 192kbps MP3\n✓ Smooth transitions with fades\n✓ Music at -12dB (perfect background level)\n✓ Normalized audio throughout\n✓ Proper silence between segments\n✓ Single combined MP3 output\n\nRESULT:\nHigh-quality professional audio ready for patients!\n"

In [2]:
import os
import subprocess

# Find FFmpeg location
try:
    result = subprocess.run(['where', 'ffmpeg'], capture_output=True, text=True)
    ffmpeg_path = result.stdout.strip().split('\n')[0]
    ffmpeg_dir = os.path.dirname(ffmpeg_path)
    
    # Set environment variables for pydub
    os.environ["PATH"] += os.pathsep + ffmpeg_dir
    
    print(f"✓ FFmpeg found at: {ffmpeg_path}")
except:
    print("⚠️  FFmpeg not found in PATH")

# NOW import pydub
from pydub import AudioSegment
from pydub.utils import which

# Manually set ffmpeg paths
AudioSegment.converter = ffmpeg_path
AudioSegment.ffmpeg = ffmpeg_path
AudioSegment.ffprobe = ffmpeg_path.replace('ffmpeg.exe', 'ffprobe.exe')

print(f"✓ Pydub configured with FFmpeg")

✓ FFmpeg found at: 
✓ Pydub configured with FFmpeg


In [5]:
"""
FIXED Audio Mixer - Proper FFmpeg Configuration
"""

import os
import subprocess
import re
from gtts import gTTS
import shutil

# STEP 1: Find FFmpeg and FFprobe
print("Configuring FFmpeg...")

try:
    # Find FFmpeg
    result = subprocess.run(['where', 'ffmpeg'], capture_output=True, text=True, shell=True)
    ffmpeg_path = result.stdout.strip().split('\n')[0]
    
    # Find FFprobe
    result2 = subprocess.run(['where', 'ffprobe'], capture_output=True, text=True, shell=True)
    ffprobe_path = result2.stdout.strip().split('\n')[0]
    
    print(f"✓ FFmpeg: {ffmpeg_path}")
    print(f"✓ FFprobe: {ffprobe_path}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    # Try conda path directly
    conda_base = os.environ.get('CONDA_PREFIX', '')
    if conda_base:
        ffmpeg_path = os.path.join(conda_base, 'Library', 'bin', 'ffmpeg.exe')
        ffprobe_path = os.path.join(conda_base, 'Library', 'bin', 'ffprobe.exe')
        print(f"Trying conda path: {ffmpeg_path}")

# STEP 2: Import pydub and configure
from pydub import AudioSegment
from pydub.effects import normalize
import pydub.utils

# STEP 3: Manually set the paths
AudioSegment.converter = ffmpeg_path
AudioSegment.ffmpeg = ffmpeg_path
AudioSegment.ffprobe = ffprobe_path
pydub.utils.which = lambda name: ffmpeg_path if 'ffmpeg' in name else ffprobe_path

print("✓ Pydub configured\n")

class ProfessionalAudioMixer:
    """Professional audio mixing"""
    
    def __init__(self, music_folder="./music", output_folder="./final_audio"):
        self.music_folder = music_folder
        self.output_folder = output_folder
        self.temp_folder = os.path.join(output_folder, "temp")
        
        os.makedirs(output_folder, exist_ok=True)
        os.makedirs(music_folder, exist_ok=True)
        os.makedirs(self.temp_folder, exist_ok=True)
        
        print("="*60)
        print("PROFESSIONAL AUDIO MIXER")
        print("="*60)
    
    def parse_narrative_with_cues(self, text):
        pattern = r'\[MUSIC CUE: Play "([^"]+)" by ([^\]]+)\]'
        parts = []
        last_end = 0
        
        for match in re.finditer(pattern, text):
            if match.start() > last_end:
                narration = text[last_end:match.start()].strip()
                if narration:
                    parts.append({'type': 'narration', 'content': narration})
            
            parts.append({
                'type': 'music',
                'song': match.group(1),
                'artist': match.group(2),
                'duration': 15
            })
            last_end = match.end()
        
        if last_end < len(text):
            remaining = text[last_end:].strip()
            if remaining:
                parts.append({'type': 'narration', 'content': remaining})
        
        return parts
    
    def find_music_file(self, song, artist):
        if not os.path.exists(self.music_folder):
            return None
        
        for file in os.listdir(self.music_folder):
            if file.endswith('.mp3'):
                if song.lower() in file.lower() or artist.lower() in file.lower():
                    filepath = os.path.join(self.music_folder, file)
                    print(f"  ✓ Found: {file}")
                    return filepath
        
        print(f"  ✗ Not found: {song}")
        return None
    
    def generate_tts_audio(self, text, temp_file):
        print(f"  Generating TTS ({len(text)} chars)...")
        
        try:
            tts = gTTS(text=text, lang='en', slow=False)
            tts.save(temp_file)
            
            # Load with explicit codec
            audio = AudioSegment.from_file(temp_file, format="mp3")
            audio = normalize(audio)
            
            print(f"  ✓ Generated: {len(audio)/1000:.1f}s")
            return audio
        except Exception as e:
            print(f"  ✗ TTS Error: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def load_music_segment(self, music_file, duration):
        try:
            print(f"  Loading: {os.path.basename(music_file)}")
            
            # Load with explicit format
            music = AudioSegment.from_file(music_file, format="mp3")
            music_clip = music[:duration * 1000]
            music_clip = music_clip - 12
            music_clip = music_clip.fade_in(2000).fade_out(2000)
            
            print(f"  ✓ Prepared: {duration}s at -12dB")
            return music_clip
        except Exception as e:
            print(f"  ✗ Music Error: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def mix_audio_segments(self, parts, output_name="motivation"):
        print("\n" + "="*60)
        print("GENERATING SEGMENTS")
        print("="*60)
        
        audio_segments = []
        
        for i, part in enumerate(parts, 1):
            print(f"\n[Segment {i}/{len(parts)}]")
            
            if part['type'] == 'narration':
                temp_file = os.path.join(self.temp_folder, f"narration_{i}.mp3")
                audio = self.generate_tts_audio(part['content'], temp_file)
                if audio:
                    audio_segments.append(audio)
                    print(f"  → Added to mix")
            
            elif part['type'] == 'music':
                music_file = self.find_music_file(part['song'], part['artist'])
                if music_file:
                    audio = self.load_music_segment(music_file, part['duration'])
                    if audio:
                        audio_segments.append(audio)
                        print(f"  → Added to mix")
        
        if not audio_segments:
            print("\n✗ No segments generated!")
            return None
        
        print("\n" + "="*60)
        print(f"COMBINING {len(audio_segments)} SEGMENTS")
        print("="*60)
        
        final_audio = AudioSegment.silent(duration=500)
        
        for i, audio in enumerate(audio_segments, 1):
            print(f"  [{i}/{len(audio_segments)}] Adding {len(audio)/1000:.1f}s...")
            final_audio += audio
            final_audio += AudioSegment.silent(duration=800)
        
        print("\n  Normalizing final audio...")
        final_audio = normalize(final_audio)
        
        output_file = os.path.join(self.output_folder, f"{output_name}.mp3")
        print(f"  Exporting to: {output_name}.mp3")
        
        final_audio.export(
            output_file, 
            format="mp3", 
            bitrate="192k",
            parameters=["-q:a", "2"]
        )
        
        # Cleanup
        try:
            shutil.rmtree(self.temp_folder)
        except:
            pass
        
        file_size = os.path.getsize(output_file) / (1024 * 1024)
        duration = len(final_audio) / 1000
        
        print("\n" + "="*60)
        print("✓ SUCCESS!")
        print("="*60)
        print(f"\nFile: {output_file}")
        print(f"Size: {file_size:.1f} MB")
        print(f"Duration: {duration/60:.1f} minutes ({duration:.0f}s)")
        print(f"Quality: 192kbps MP3")
        print(f"\n✓ Ready to play!")
        
        return output_file

# USAGE
narrative = """**DAILY EXERCISE SCHEDULE:**

Morning (7:30 AM): Cat-Cow Stretch, Pelvic Tilts, Walking
Afternoon (2:00 PM): Bird-Dog Exercise, Seated Twists
Evening (7:00 PM): Child's Pose, Hamstring Stretches

**MOTIVATIONAL NARRATIVE:**

[MUSIC CUE: Play "Eye of the Tiger" by Survivor]

Good morning, John. Today is a new chapter in your recovery journey. Every rep you do today is building toward a pain-free future.

[MUSIC CUE: Play "Stronger" by Kanye West]

Think about your children, John. Think about playing in the backyard without pain. That's where we're headed.

[MUSIC CUE: Play "Don't Stop Believin'" by Journey]

Three months from now, you'll realize this was the turning point. Welcome to your recovery, John."""

mixer = ProfessionalAudioMixer()
parts = mixer.parse_narrative_with_cues(narrative)

print(f"\nParsed {len(parts)} segments")
for i, p in enumerate(parts, 1):
    if p['type'] == 'narration':
        print(f"  {i}. Narration ({len(p['content'])} chars)")
    else:
        print(f"  {i}. Music: {p['song']}")

output = mixer.mix_audio_segments(parts, "john_motivation_professional")

Configuring FFmpeg...
✓ FFmpeg: 
✓ FFprobe: 
✓ Pydub configured

PROFESSIONAL AUDIO MIXER

Parsed 7 segments
  1. Narration (223 chars)
  2. Music: Eye of the Tiger
  3. Narration (130 chars)
  4. Music: Stronger
  5. Narration (109 chars)
  6. Music: Don't Stop Believin'
  7. Narration (97 chars)

GENERATING SEGMENTS

[Segment 1/7]
  Generating TTS (223 chars)...


c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
Traceback (most recent call last):
  File "C:\Users\drars\AppData\Local\Temp\ipykernel_15748\2039840091.py", line 112, in generate_tts_audio
    audio = AudioSegment.from_file(temp_file, format="mp3")
  File "c:\Users\drars\anaconda3\Lib\site-packages\pydub\audio_segment.py", line 728, in from_file
    info = mediainfo_json(orig_file, read_ahead_limit=read_ahead_limit)
  File "c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py", line 274, in mediainfo_json
    res = Popen(command, stdin=stdin_parameter, stdout=PIPE, stderr=PIPE)
  File "c:\Users\drars\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  ✗ TTS Error: [WinError 2] The system cannot find the file specified

[Segment 2/7]
  ✓ Found: Survivor - Eye of the Tiger.mp3
  Loading: Survivor - Eye of the Tiger.mp3
  ✗ Music Error: [WinError 2] The system cannot find the file specified

[Segment 3/7]
  Generating TTS (130 chars)...


c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
Traceback (most recent call last):
  File "C:\Users\drars\AppData\Local\Temp\ipykernel_15748\2039840091.py", line 112, in generate_tts_audio
    audio = AudioSegment.from_file(temp_file, format="mp3")
  File "c:\Users\drars\anaconda3\Lib\site-packages\pydub\audio_segment.py", line 728, in from_file
    info = mediainfo_json(orig_file, read_ahead_limit=read_ahead_limit)
  File "c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py", line 274, in mediainfo_json
    res = Popen(command, stdin=stdin_parameter, stdout=PIPE, stderr=PIPE)
  File "c:\Users\drars\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  ✗ TTS Error: [WinError 2] The system cannot find the file specified

[Segment 4/7]
  ✓ Found: Kanye West - Stronger.mp3
  Loading: Kanye West - Stronger.mp3
  ✗ Music Error: [WinError 2] The system cannot find the file specified

[Segment 5/7]
  Generating TTS (109 chars)...


c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
Traceback (most recent call last):
  File "C:\Users\drars\AppData\Local\Temp\ipykernel_15748\2039840091.py", line 112, in generate_tts_audio
    audio = AudioSegment.from_file(temp_file, format="mp3")
  File "c:\Users\drars\anaconda3\Lib\site-packages\pydub\audio_segment.py", line 728, in from_file
    info = mediainfo_json(orig_file, read_ahead_limit=read_ahead_limit)
  File "c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py", line 274, in mediainfo_json
    res = Popen(command, stdin=stdin_parameter, stdout=PIPE, stderr=PIPE)
  File "c:\Users\drars\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  ✗ TTS Error: [WinError 2] The system cannot find the file specified

[Segment 6/7]
  ✓ Found: Journey - Don't Stop Believin'.mp3
  Loading: Journey - Don't Stop Believin'.mp3
  ✗ Music Error: [WinError 2] The system cannot find the file specified

[Segment 7/7]
  Generating TTS (97 chars)...
  ✗ TTS Error: [WinError 2] The system cannot find the file specified

✗ No segments generated!


c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
Traceback (most recent call last):
  File "C:\Users\drars\AppData\Local\Temp\ipykernel_15748\2039840091.py", line 112, in generate_tts_audio
    audio = AudioSegment.from_file(temp_file, format="mp3")
  File "c:\Users\drars\anaconda3\Lib\site-packages\pydub\audio_segment.py", line 728, in from_file
    info = mediainfo_json(orig_file, read_ahead_limit=read_ahead_limit)
  File "c:\Users\drars\anaconda3\Lib\site-packages\pydub\utils.py", line 274, in mediainfo_json
    res = Popen(command, stdin=stdin_parameter, stdout=PIPE, stderr=PIPE)
  File "c:\Users\drars\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [2]:
# Install spotdl
!pip install spotdl

# Download songs
spotdl "Eye of the Tiger"
spotdl "Stronger Kanye West"
spotdl "Don't Stop Believin'"
spotdl "Hall of Fame The Script"
spotdl "Lose Yourself Eminem"
spotdl "Titanium David Guetta"
spotdl "Roar Katy Perry"
spotdl "Fight Song Rachel Platten"
spotdl "Unwritten Natasha Bedingfield"
spotdl "Brave Sara Bareilles"
spotdl "Till I Collapse Eminem"
spotdl "Remember the Name Fort Minor"

# Move all to music folder
mkdir music
move *.mp3 music/

SyntaxError: invalid syntax (1089930918.py, line 5)